In [ ]:
# Cell 1 — install llama.cpp server (prebuilt CUDA binary) and download a small GGUF model
!apt-get update -qq && apt-get install -qq -y curl
!curl -L -o llama-server.tar.gz https://github.com/ggerganov/llama.cpp/releases/latest/download/llama-b1-bin-ubuntu-cuda-cu12.4.1-x64.tar.gz || echo 'DIRECT_BINARY_FAILED_WILL_BUILD_FROM_SOURCE'
!pip install -q huggingface_hub
from huggingface_hub import hf_hub_download
model_path = hf_hub_download(
    repo_id='Qwen/Qwen2.5-1.5B-Instruct-GGUF',
    filename='qwen2.5-1.5b-instruct-q4_k_m.gguf'
)
print('MODEL_PATH:', model_path)

In [ ]:
# Cell 2 — build llama.cpp WITH CUDA support and start the server with mlock disabled.
# v3 attempt failed to build from source; this version adds real diagnostics (nvcc/cmake
# presence, full build log written to a file with the actual error surfaced) instead of
# blindly retrying, so we know the real cause if it fails again.
import os
os.environ['CMAKE_ARGS'] = '-DGGML_CUDA=on'
os.environ['FORCE_CMAKE'] = '1'
os.environ['CMAKE_BUILD_PARALLEL_LEVEL'] = '4'

print('--- CUDA toolchain check ---')
!which nvcc && nvcc --version || echo 'NVCC_NOT_FOUND'
!which cmake && cmake --version || echo 'CMAKE_NOT_FOUND'
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'NVIDIA_SMI_FAILED'

print('--- building llama-cpp-python (full log -> build.log) ---')
!pip install -q --no-cache-dir --force-reinstall llama-cpp-python[server] > build.log 2>&1

with open('build.log') as f:
    lines = f.readlines()
print(f'build.log has {len(lines)} lines')

error_lines = [i for i, l in enumerate(lines) if "error" in l.lower()]
if error_lines:
    print(f'--- context around first error (line {error_lines[0]}) ---')
    start = max(0, error_lines[0] - 20)
    end = min(len(lines), error_lines[0] + 10)
    print(''.join(lines[start:end]))
else:
    print("--- no error string found, last 60 lines ---")
    print(''.join(lines[-60:]))

import importlib
try:
    importlib.import_module('llama_cpp')
    print('IMPORT_OK: llama_cpp module is importable')
except ImportError as e:
    print(f'IMPORT_FAILED: {e}')

import subprocess, time, requests

server_proc = subprocess.Popen([
    'python', '-m', 'llama_cpp.server',
    '--model', model_path,
    '--host', '0.0.0.0',
    '--port', '8000',
    '--n_gpu_layers', '-1',
    '--use_mlock', 'false',
])

print('SERVER_STARTING pid=', server_proc.pid)
ready = False
for i in range(30):
    if server_proc.poll() is not None:
        print('SERVER_PROCESS_EXITED code=', server_proc.returncode)
        break
    try:
        resp = requests.get('http://localhost:8000/v1/models', timeout=3)
        if resp.status_code == 200:
            print(f'SERVER_READY after {i * 10}s')
            ready = True
            break
    except Exception:
        pass
    print(f'  ...waiting for server ({i * 10}s elapsed)')
    time.sleep(10)

if not ready:
    print('SERVER_NOT_READY after 300s -- check build.log / errors above')


In [ ]:
# Cell 3 — start cloudflared tunnel, print the public URL.
# Only keep the session alive forever (the tunnel keep-alive loop) if the server
# actually started in cell 2. Otherwise a failed run would idle for the full 9-hour
# batch session cap doing nothing useful, occupying one of the 2 concurrent GPU
# session slots -- this is exactly what happened with the v2/v3 pushes.
if not globals().get('ready', False):
    print('SKIPPING_TUNNEL: server was not ready after cell 2, exiting so this session frees up promptly')
else:
    !curl -L -o cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x cloudflared
    import subprocess, time, re
    tunnel_proc = subprocess.Popen(
        ['./cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    url = None
    for _ in range(60):
        line = tunnel_proc.stdout.readline()
        print(line, end='')
        match = re.search(r'https://[a-zA-Z0-9\-]+\.trycloudflare\.com', line)
        if match:
            url = match.group(0)
            break
    print()
    print('TUNNEL_URL:', url)
    print('Keep this cell running -- closing it kills the tunnel.')
    while True:
        time.sleep(60)
